# 🐦‍⬛ Karga-2B-Thinking: Colab Quickstart
Bu not defteri, `ilkayO/Karga-2B-Thinking` modelini ücretsiz bir Google Colab T4 GPU üzerinde hızlıca test etmeniz için hazırlanmıştır.

In [ ]:
# @title ⚙️ 1. Kurulum (Install Dependencies)
# Bu hücreyi çalıştırarak gerekli kütüphaneleri indirin.
!pip install -q -U transformers torch accelerate

In [ ]:
# @title 🧠 2. Karga-2B-Thinking Modelini Yükle
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "ilkayO/Karga-2B-Thinking"

print("Tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Model VRAM'e indiriliyor ve yükleniyor (Yaklaşık 4GB)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
print("\n✅ Yükleme Tamamlandı! Model kullanıma hazır.")

In [ ]:
# @title 💬 3. Karga'ya Soru Sor
soru = "Ahmet'in 3 elması var. Ayşe, Ahmet'e elmalarının 2 katı kadar daha elma verdi. Daha sonra Ahmet elmalarının 4 tanesini yedi. Ahmet'in kaç elması kaldı? Adım adım düşün." # @param {type:"string"}

messages = [
    {"role": "system", "content": "Adın Karga. Soruları mantıklı ve adım adım düşünerek yanıtla."},
    {"role": "user", "content": soru}
]

inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)

print("Karga düşünüyor...\n" + "="*50)

with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_new_tokens=1024,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True
    )

cevap = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(cevap)